# Test Session Class with Synthetic Data

This notebook tests the Session class methods using controlled synthetic data to verify functionality.

In [1]:
import pandas as pd
import numpy as np
import holoviews as hv
import hvplot.pandas
import panel as pn

from holoviews import opts
from bokeh.io import output_notebook

# Import the session analysis class with reload capability
import importlib
import session_class
importlib.reload(session_class)
from session_class import Session

print("Session class imported successfully!")

output_notebook()
hv.extension('bokeh')

Session class imported successfully!


Loading BokehJS ...

## Test 1: Simple 2-Cell Session

Create a session with 2 cells, each with a GO trial:
- Cell 1: spike at go_cue (100ms)
- Cell 2: spike 100ms after go_cue (200ms)

Visualize using spike counts heatmap.

In [2]:
def create_simple_session_data():
    """
    Create simple session with 2 cells:
    - Cell 1: spike at go_cue (100ms)
    - Cell 2: spike 100ms after go_cue (200ms)
    """
    test_session = 'test_session_simple'
    
    trials = [
        # Cell 1: GO trial with spike at go_cue
        {
            'cell_ID': 1,
            'cell_type': 'MSN',
            'trial_session': test_session,
            'type': 'GO',
            'dir': 0,
            'trial_failed': False,
            'go_cue': 100,
            'stop_cue': np.nan,
            'first_relevant_saccade': 250,
            'ssd_number': np.nan,
            'neural_data': [100],  # Spike at go_cue
            'trial_number': 1,
        },
        # Cell 2: GO trial with spike 100ms after go_cue
        {
            'cell_ID': 2,
            'cell_type': 'MSN',
            'trial_session': test_session,
            'type': 'GO',
            'dir': 0,
            'trial_failed': False,
            'go_cue': 100,
            'stop_cue': np.nan,
            'first_relevant_saccade': 250,
            'ssd_number': np.nan,
            'neural_data': [200],  # Spike 100ms after go_cue
            'trial_number': 1,
        },
    ]
    
    return pd.DataFrame(trials)

# Create session
simple_df = create_simple_session_data()
simple_session = Session(simple_df, verbose=True)

# Visualize with spike counts heatmap
simple_session.plot_population_spike_counts_heatmap(
    epok=[-50, 150],
    bin_size=1,
    alignment_point='go_cue',
    trial_type='GO',
    normalize=False,
    sort_by_peak=True
)

Session test_session_simple initialized:
  - Number of cells: 2
  - Total trials: 1
  - Trial types: ['GO']
  - Directions: [np.int64(0)]


:HeatMap   [columns,index]   (value)

## Test 2: 20-Cell Session with Direction-Selective Activity

Create a session with 20 cells, each with 2 GO trials (left and right):
- **GO Left (dir=180)**: Cell i spikes at 90+i ms (before go_cue)
- **GO Right (dir=0)**: Cell i spikes at 110-i ms (after go_cue)
- **go_cue**: 100ms for all trials

This creates a gradient pattern where:
- Left: Earlier cells spike first (cell 0 at 90ms, cell 19 at 109ms)
- Right: Later cells spike first (cell 19 at 91ms, cell 0 at 110ms)

In [3]:
def create_gradient_session_data(n_cells=20):
    """
    Create session with direction-selective gradient activity.
    
    Parameters:
    -----------
    n_cells : int
        Number of cells (default: 20)
    
    Returns:
    --------
    pd.DataFrame with trial data for all cells
    """
    test_session = 'test_session_gradient'
    go_cue_time = 100
    
    trials = []
    
    for cell_id in range(n_cells):
        # GO Left trial: cell i spikes at 90+i
        trials.append({
            'cell_ID': cell_id,
            'cell_type': 'MSN',
            'trial_session': test_session,
            'type': 'GO',
            'dir': 180,  # Left
            'trial_failed': False,
            'go_cue': go_cue_time,
            'stop_cue': np.nan,
            'first_relevant_saccade': 250,
            'ssd_number': np.nan,
            'neural_data': [90 + cell_id],  # Spike at 90+i
            'trial_number': cell_id * 2 + 1,
        })
        
        # GO Right trial: cell i spikes at 110-i
        trials.append({
            'cell_ID': cell_id,
            'cell_type': 'MSN',
            'trial_session': test_session,
            'type': 'GO',
            'dir': 0,  # Right
            'trial_failed': False,
            'go_cue': go_cue_time,
            'stop_cue': np.nan,
            'first_relevant_saccade': 250,
            'ssd_number': np.nan,
            'neural_data': [110 - cell_id],  # Spike at 110-i
            'trial_number': cell_id * 2 + 2,
        })
    
    return pd.DataFrame(trials)

# Create session
gradient_df = create_gradient_session_data(n_cells=20)
gradient_session = Session(gradient_df, verbose=True)

print(f"\nCreated session with {gradient_session.n_cells} cells and {gradient_session.n_trials} trials")

Session test_session_gradient initialized:
  - Number of cells: 20
  - Total trials: 40
  - Trial types: ['GO']
  - Directions: [np.int64(0), np.int64(180)]

Created session with 20 cells and 40 trials


### Visualize Left Direction (dir=180)

In [4]:
# Visualize LEFT direction
gradient_session.plot_population_spike_counts_heatmap(
    epok=[-20, 30],
    bin_size=1,
    alignment_point='go_cue',
    trial_type='GO',
    direction=180,  # Left
    normalize=False,
    sort_by_peak=True
)

:HeatMap   [columns,index]   (value)

### Visualize Right Direction (dir=0)

In [5]:
# Visualize RIGHT direction
gradient_session.plot_population_spike_counts_heatmap(
    epok=[-20, 30],
    bin_size=1,
    alignment_point='go_cue',
    trial_type='GO',
    direction=0,  # Right
    normalize=False,
    sort_by_peak=True
)

:HeatMap   [columns,index]   (value)

### Compare Left vs Right Side-by-Side

Use the plotting method to create both heatmaps with matching cell order.

In [ ]:
# Create side-by-side comparison
left_plot = gradient_session.plot_population_spike_counts_heatmap(
    epok=[-20, 30],
    bin_size=1,
    alignment_point='go_cue',
    trial_type='GO',
    direction=180,
    normalize=False,
    sort_by_peak=True
).opts(width=400, title='Left (180°) - Gradient 90+i')

right_plot = gradient_session.plot_population_spike_counts_heatmap(
    epok=[-20, 30],
    bin_size=1,
    alignment_point='go_cue',
    trial_type='GO',
    direction=0,
    normalize=False,
    sort_by_peak=F
).opts(width=400, title='Right (0°) - Gradient 110-i')

(left_plot + right_plot).cols(2)

:Layout
   .HeatMap.I  :HeatMap   [columns,index]   (value)
   .HeatMap.II :HeatMap   [columns,index]   (value)

### Verify Raw Data

Check the spike counts for a few cells to confirm the pattern.

In [7]:
# Check spike counts for cells 0, 5, 10, 15, 19
test_cells = [0, 5, 10, 15, 19]

print("Left trials (dir=180): Cell i spikes at 90+i ms")
for cell_id in test_cells:
    bins, counts, n_trials = gradient_session.get_cell_spike_counts(
        cell_id=cell_id,
        epok=[-20, 30],
        bin_size=1,
        alignment_point='go_cue',
        trial_type='GO',
        direction=180,
        normalize=False
    )
    if bins is not None:
        spike_time = bins[counts > 0][0] if any(counts > 0) else None
        expected_time = 90 + cell_id - 100  # Relative to go_cue at 100
        print(f"  Cell {cell_id}: spike at {spike_time}ms (expected: {expected_time}ms)")

print("\nRight trials (dir=0): Cell i spikes at 110-i ms")
for cell_id in test_cells:
    bins, counts, n_trials = gradient_session.get_cell_spike_counts(
        cell_id=cell_id,
        epok=[-20, 30],
        bin_size=1,
        alignment_point='go_cue',
        trial_type='GO',
        direction=0,
        normalize=False
    )
    if bins is not None:
        spike_time = bins[counts > 0][0] if any(counts > 0) else None
        expected_time = 110 - cell_id - 100  # Relative to go_cue at 100
        print(f"  Cell {cell_id}: spike at {spike_time}ms (expected: {expected_time}ms)")

Left trials (dir=180): Cell i spikes at 90+i ms
  Cell 0: spike at -10ms (expected: -10ms)
  Cell 5: spike at -5ms (expected: -5ms)
  Cell 10: spike at 0ms (expected: 0ms)
  Cell 15: spike at 5ms (expected: 5ms)
  Cell 19: spike at 9ms (expected: 9ms)

Right trials (dir=0): Cell i spikes at 110-i ms
  Cell 0: spike at 10ms (expected: 10ms)
  Cell 5: spike at 5ms (expected: 5ms)
  Cell 10: spike at 0ms (expected: 0ms)
  Cell 15: spike at -5ms (expected: -5ms)
  Cell 19: spike at -9ms (expected: -9ms)
